In [2]:
import pandas as pd
import numpy as np

customer_profiles = pd.read_excel("../customer_profiles_5000.xlsx")
loan_history = pd.read_excel("../loan_history_5000.xlsx")
transaction_history = pd.read_excel("../transaction_history_5000.xlsx")

In [3]:
customer_profiles.head()

,CustomerID,FirstName,MiddleName,LastName,DateOfBirth,NumberOfDependents,LifecycleStatus,DateRegistered,Country,Address,AvgRiskScore,Defaulted
0,CUST000000,Dustin,Jennifer,Smith,1972-01-12,0,Dormant,2021-07-28,Ethiopia,"354 Morgan Passage Apt. 003, Smithstad, HI 66030",0.440000,0
1,CUST000001,Spencer,Edward,Montoya,1956-10-22,0,Active,2021-12-27,Tanzania,"47184 Tammie Glens, West Dorothy, MD 78937",0.843939,1
2,CUST000002,Kelli,Joseph,Aguirre,1971-11-23,3,Active,2024-03-22,Ethiopia,"Unit 8246 Box 8996, DPO AA 98852",0.633333,1
3,CUST000003,Janice,James,Huynh,1960-01-12,0,Active,2023-10-01,Kenya,"81564 Mason Bridge, Elizabethbury, HI 80276",0.825000,1
4,CUST000004,Jessica,Christopher,Estrada,2006-09-29,2,Active,2022-08-31,Kenya,"PSC 8041, Box 5304, APO AA 01202",0.608301,1


In [4]:
loan_history.head()

,LoanId,CustomerID,Amount,NumberOfEMIs,LateEMIs,LoanStatus,RiskScore
0,ee407b7a-ee3e-41b6-a07f-d4e3579ae94b,CUST003482,2608.10,34,14,Active,0.711765
1,4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c,CUST003482,4694.55,13,6,Closed,0.461538
2,4cdfabca-b104-41f6-bdf8-efc86be36f00,CUST001248,4108.84,30,14,Active,0.766667
3,4531d63f-dc19-43fe-9c6e-5ce83bf37870,CUST002364,7998.41,23,6,Defaulted,1.260870
4,419a6a3d-7852-4aa6-9570-25abd0520e0a,CUST002364,5256.21,24,0,Closed,0.000000


In [5]:
transaction_history.head()

,TransactionId,AccountId,CustomerID,TransactionDate,Amount,Type
0,baef0952-03ec-45d9-9a95-1fb5958b286a,1f749c82-af80-44b6-b4f0-bc7306eb47a3,CUST000000,2024-04-11,463.58,Incoming
1,7336454a-b50c-4acf-983d-ebf9d3e46ea9,60b0069b-8d1b-4ada-98db-53db5a5155ba,CUST000000,2024-04-14,475.84,Outgoing
2,8e19a2db-32ce-4d81-9ab1-32a3667206bb,08e3a401-de78-4e6e-8dd8-82a08eb8f87c,CUST000000,2024-05-25,467.54,Outgoing
3,6a03b7d4-7540-4a6f-bd00-e0cbd93e1d87,7916b8e6-3142-48a2-8fd0-bdb67c91dc9b,CUST000000,2025-05-03,755.63,Incoming
4,5e5d8cf1-6125-4807-a5e8-e67b0d82bce2,d39aab00-0332-4b58-b4ec-027d9102b924,CUST000000,2023-11-01,636.82,Outgoing


# Generate Loan Application Date

In [6]:
customer_profiles['DateRegistered'] = pd.to_datetime(customer_profiles['DateRegistered'])
loan_history = loan_history.merge(customer_profiles[['CustomerID', 'DateRegistered']], on='CustomerID', how='left')
max_days_after_registration = 730
np.random.seed(42)  
loan_history['DaysAfterRegistration'] = np.random.randint(1, max_days_after_registration + 1, size=len(loan_history))
loan_history['LoanApplicationDate'] = loan_history['DateRegistered'] + pd.to_timedelta(loan_history['DaysAfterRegistration'], unit='D')
loan_history.drop(columns=['DaysAfterRegistration','DateRegistered'], inplace=True)

In [7]:
loan_history

,LoanId,CustomerID,Amount,NumberOfEMIs,LateEMIs,LoanStatus,RiskScore,LoanApplicationDate
0,ee407b7a-ee3e-41b6-a07f-d4e3579ae94b,CUST003482,2608.10,34,14,Active,0.711765,2022-05-17
1,4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c,CUST003482,4694.55,13,6,Closed,0.461538,2023-04-15
2,4cdfabca-b104-41f6-bdf8-efc86be36f00,CUST001248,4108.84,30,14,Active,0.766667,2024-04-08
3,4531d63f-dc19-43fe-9c6e-5ce83bf37870,CUST002364,7998.41,23,6,Defaulted,1.260870,2022-01-02
4,419a6a3d-7852-4aa6-9570-25abd0520e0a,CUST002364,5256.21,24,0,Closed,0.000000,2021-11-28
...,...,...,...,...,...,...,...,...
6959,43dec1a5-024e-455d-ba82-712daf34b38a,CUST002230,2378.64,16,4,Active,0.550000,2023-10-25
6960,aeabe1a6-8f74-4aff-b0a9-b793a57d8934,CUST004020,4283.90,36,11,Defaulted,1.305556,2023-02-27
6961,725c8eae-f737-419c-99ec-0ae6762fcff1,CUST000274,4500.38,35,15,Defaulted,1.428571,2021-10-12
6962,b8125e42-8009-419f-a061-2047da6f81f2,CUST002648,2608.55,17,0,Active,0.300000,2022-07-07


# Target Engineering

In [8]:
loan_history.LoanStatus.value_counts()

LoanStatus
Defaulted    2421
Active       2305
Closed       2238
Name: count, dtype: int64

## Default Classification Analysis

We observe that we have:
- **2,421 defaulted** loans
- **2,238 non-defaulted** loans (these are marked as "closed")
- **2,305 active loans** loans that not closed but not defaulted also 
 
To refine our assumptions, we can use a rule based on the number of **Late EMIs (Equated Monthly Installments)**:

> If `LateEMIs >= 3`, we can tentatively classify the loan as *defaulted*, aligning with the standard provided by the Bank for International Settlements (BIS).

### International Definition of Default

According to the **Bank for International Settlements (BIS)**:

> "A loan is in default when the borrower is past due more than **90 days** on any material credit obligation to the bank."

---

### Note on Data Quality

Since this dataset is **simulated**, and the `LoanApplicationDate` is also generated artificially, there may be **inaccuracies or inconsistencies** in the data. However, this is acceptable for our current **experimental purposes**.

In production with **real data**, we expect to work with **well-structured and high-quality information**.


In [9]:
loan_history.LoanStatus = np.where(loan_history.LoanStatus == 'Active', 0, loan_history.LoanStatus)
loan_history.LoanStatus = np.where(loan_history.LoanStatus == 'Closed', 0, loan_history.LoanStatus)
loan_history.LoanStatus = np.where(loan_history.LoanStatus == 'Defaulted', 1, loan_history.LoanStatus)

In [10]:
loan_history.LoanStatus.value_counts(),loan_history.LoanStatus.value_counts(1)

(LoanStatus
 0    4543
 1    2421
 Name: count, dtype: int64,
 LoanStatus
 0    0.652355
 1    0.347645
 Name: proportion, dtype: float64)

In [11]:
loan_history.rename(columns={'LoanStatus':'Target'},inplace=True)

In [12]:
loan_history

,LoanId,CustomerID,Amount,NumberOfEMIs,LateEMIs,Target,RiskScore,LoanApplicationDate
0,ee407b7a-ee3e-41b6-a07f-d4e3579ae94b,CUST003482,2608.10,34,14,0,0.711765,2022-05-17
1,4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c,CUST003482,4694.55,13,6,0,0.461538,2023-04-15
2,4cdfabca-b104-41f6-bdf8-efc86be36f00,CUST001248,4108.84,30,14,0,0.766667,2024-04-08
3,4531d63f-dc19-43fe-9c6e-5ce83bf37870,CUST002364,7998.41,23,6,1,1.260870,2022-01-02
4,419a6a3d-7852-4aa6-9570-25abd0520e0a,CUST002364,5256.21,24,0,0,0.000000,2021-11-28
...,...,...,...,...,...,...,...,...
6959,43dec1a5-024e-455d-ba82-712daf34b38a,CUST002230,2378.64,16,4,0,0.550000,2023-10-25
6960,aeabe1a6-8f74-4aff-b0a9-b793a57d8934,CUST004020,4283.90,36,11,1,1.305556,2023-02-27
6961,725c8eae-f737-419c-99ec-0ae6762fcff1,CUST000274,4500.38,35,15,1,1.428571,2021-10-12
6962,b8125e42-8009-419f-a061-2047da6f81f2,CUST002648,2608.55,17,0,0,0.300000,2022-07-07


# Feature Calculations

## Assumption on Loan Application Data

Since we **do not have actual loan application data**, and only possess **loan history**, we make the following assumption:

> Each record in the loan history dataset is treated as an individual **loan application**.

This allows us to **calculate features** as if they are based on loan application data, even though we are deriving them from the same dataset.

---

### Important Caveat

This approach is **not ideal**, and the dataset is **poorly simulated** in this regard.

In a real-world scenario:
- Banks and credit institutions **always store loan application data and loan history separately**.
- Loan history may include loans from **multiple banks**, while the loan application dataset is a distinct and structured entity.

We proceed this way **only because** the application data is missing in our current simulation.

---

### Production Considerations

In production environments with real data:
- We will **identify and address** this structural issue.
- We will explicitly **request access to loan application data** to ensure accurate feature engineering and modeling.


In [13]:
df = loan_history.copy()

In [14]:
df['LoanApplicationDate'] = pd.to_datetime(df['LoanApplicationDate'])

df = df.sort_values(by=['CustomerID', 'LoanApplicationDate'])

## Loan history features

In [15]:
history_features = []

for idx, row in df.iterrows():
    cust_id = row['CustomerID']
    app_date = row['LoanApplicationDate']
    
    # History = all previous loans of this customer
    history = df[
        (df['CustomerID'] == cust_id) &
        (df['LoanApplicationDate'] < app_date)
    ]
    
    count_loans = len(history)
    total_amt = history['Amount'].sum()
    avg_amt = history['Amount'].mean() if not history.empty else 0
    std_amt = history['Amount'].std() if len(history) > 1 else 0
    min_amt = history['Amount'].min() if not history.empty else 0
    max_amt = history['Amount'].max() if not history.empty else 0
    
    sum_emis = history['NumberOfEMIs'].sum()
    mean_emis = history['NumberOfEMIs'].mean() if not history.empty else 0
    
    default_count = history[history['Status'] == 'Defaulted'].shape[0] if 'Status' in df.columns else 0
    closed_count = history[history['Status'] == 'Closed'].shape[0] if 'Status' in df.columns else 0
    active_count = history[history['Status'] == 'Active'].shape[0] if 'Status' in df.columns else 0
    
    default_ratio = default_count / count_loans if count_loans > 0 else 0
    total_amt_per_loan = total_amt / count_loans if count_loans > 0 else 0
    
    if not history.empty:
        last_loan_date = history['LoanApplicationDate'].max()
        first_loan_date = history['LoanApplicationDate'].min()
        days_since_last = (app_date - last_loan_date).days
        months_since_first = (app_date - first_loan_date).days // 30
        time_diffs = history['LoanApplicationDate'].sort_values().diff().dropna().dt.days
        avg_loan_gap = time_diffs.mean() if not time_diffs.empty else None
        std_time_diffs = time_diffs.std() if not time_diffs.empty else None
    else:
        days_since_last = None
        months_since_first = None
        avg_loan_gap = None
        std_time_diffs = None

    short_term_loans = history[history['NumberOfEMIs'] <= 6].shape[0]
    short_term_ratio = short_term_loans / count_loans if count_loans > 0 else 0
    loans_per_year = count_loans / (months_since_first / 12) if months_since_first and months_since_first > 0 else 0

    history_features.append({
        'LoanId': row['LoanId'],
        'CustomerID': cust_id,
        'Hist_CountLoans': count_loans,
        'Hist_TotalAmount': total_amt,
        'Hist_AvgAmount': avg_amt,
        'Hist_StdAmount': std_amt,
        'Hist_MinAmount': min_amt,
        'Hist_MaxAmount': max_amt,
        'Hist_SumEMIs': sum_emis,
        'Hist_MeanEMIs': mean_emis,
        'Hist_DefaultedCount': default_count,
        'Hist_ClosedLoansCount': closed_count,
        'Hist_ActiveLoansCount': active_count,
        'Hist_DefaultRatio': default_ratio,
        'Hist_TotalAmountPerLoan': total_amt_per_loan,
        'Hist_DaysSinceLastLoan': days_since_last,
        'Hist_MonthsSinceFirstLoan': months_since_first,
        'Hist_AvgLoanGap': avg_loan_gap,
        'Hist_TimeDiffsStd': std_time_diffs,
        'Hist_LoanFrequencyPerYear': loans_per_year,
        'Hist_ShortTermLoanShare': short_term_ratio
    })

history_df = pd.DataFrame(history_features)

### check if all correct

In [16]:
loan_history[loan_history.CustomerID == 'CUST003482']

,LoanId,CustomerID,Amount,NumberOfEMIs,LateEMIs,Target,RiskScore,LoanApplicationDate
0,ee407b7a-ee3e-41b6-a07f-d4e3579ae94b,CUST003482,2608.10,34,14,0,0.711765,2022-05-17
1,4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c,CUST003482,4694.55,13,6,0,0.461538,2023-04-15


In [17]:
history_df[history_df.LoanId == 'ee407b7a-ee3e-41b6-a07f-d4e3579ae94b']

,LoanId,CustomerID,Hist_CountLoans,Hist_TotalAmount,Hist_AvgAmount,Hist_StdAmount,Hist_MinAmount,Hist_MaxAmount,Hist_SumEMIs,Hist_MeanEMIs,...,Hist_ClosedLoansCount,Hist_ActiveLoansCount,Hist_DefaultRatio,Hist_TotalAmountPerLoan,Hist_DaysSinceLastLoan,Hist_MonthsSinceFirstLoan,Hist_AvgLoanGap,Hist_TimeDiffsStd,Hist_LoanFrequencyPerYear,Hist_ShortTermLoanShare
4786,ee407b7a-ee3e-41b6-a07f-d4e3579ae94b,CUST003482,0,0.0,0.0,0.0,0.0,0.0,0,0.0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0


In [18]:
history_df[history_df.LoanId == '4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c']

,LoanId,CustomerID,Hist_CountLoans,Hist_TotalAmount,Hist_AvgAmount,Hist_StdAmount,Hist_MinAmount,Hist_MaxAmount,Hist_SumEMIs,Hist_MeanEMIs,...,Hist_ClosedLoansCount,Hist_ActiveLoansCount,Hist_DefaultRatio,Hist_TotalAmountPerLoan,Hist_DaysSinceLastLoan,Hist_MonthsSinceFirstLoan,Hist_AvgLoanGap,Hist_TimeDiffsStd,Hist_LoanFrequencyPerYear,Hist_ShortTermLoanShare
4787,4bdf85ef-9a2d-4ddd-a48c-22fef2e7339c,CUST003482,1,2608.1,2608.1,0.0,2608.1,2608.1,34,34.0,...,0,0,0.0,2608.1,333.0,11.0,NaN,NaN,1.090909,0.0


### Merge loan history and applications

In [19]:
df = df.merge(history_df, on=['LoanId', 'CustomerID'], how='left')

### So we need to remove the information which is from the future , because at the time loan application we dont have information about NumberOfEMIs , LateEMIs and etc.

In [20]:
df.drop(columns=['NumberOfEMIs','LateEMIs','RiskScore'],inplace=True)

## Customer profile features

In the customer profile data, we will have much more information in the case of real data. It will likely include **income**, but in many countries, this information comes from a **bureau** — an organization (public or private) that collects data from all banks and employers. These bureaus can provide details about a client's **salary, income level, workplace, employment history**, and more. 

In a real-world scenario, we will likely be asked to obtain this additional information from such bureaus to improve the accuracy and completeness of our dataset.


In [21]:
loan_app_df = df[['LoanId','CustomerID','LoanApplicationDate']]

In [22]:
cust_profile_df = customer_profiles.copy()

In [23]:
loan_app_df['LoanApplicationDate'] = pd.to_datetime(loan_app_df['LoanApplicationDate'])
cust_profile_df['DateOfBirth'] = pd.to_datetime(cust_profile_df['DateOfBirth'])
cust_profile_df['DateRegistered'] = pd.to_datetime(cust_profile_df['DateRegistered'])

loan_app_df = loan_app_df.merge(cust_profile_df, on='CustomerID', how='left')


loan_app_df['CustomerAgeAtApplication'] = (loan_app_df['LoanApplicationDate'] - loan_app_df['DateOfBirth']).dt.days // 365

loan_app_df['DaysSinceRegistration'] = (loan_app_df['LoanApplicationDate'] - loan_app_df['DateRegistered']).dt.days

loan_app_df['IsRegisteredBeforeApplication'] = loan_app_df['DaysSinceRegistration'].apply(lambda x: 1 if x >= 0 else 0)

loan_app_df['LifecycleStatus_Active'] = (loan_app_df['LifecycleStatus'] == 'Active').astype(int)
loan_app_df['LifecycleStatus_Dormant'] = (loan_app_df['LifecycleStatus'] == 'Dormant').astype(int)
loan_app_df['LifecycleStatus_Closed'] = (loan_app_df['LifecycleStatus'] == 'Closed').astype(int)

loan_app_df['PreviouslyDefaulted'] = loan_app_df['Defaulted']  # Assuming 1 = Yes, 0 = No

loan_app_df['IsFromKenya'] = (loan_app_df['Country'] == 'Kenya').astype(int)
loan_app_df['IsFromEthiopia'] = (loan_app_df['Country'] == 'Ethiopia').astype(int)
loan_app_df['IsFromUganda'] = (loan_app_df['Country'] == 'Uganda').astype(int)

loan_app_df['AddressLength'] = loan_app_df['Address'].astype(str).apply(len)

loan_app_df['ApplicationMonth'] = loan_app_df['LoanApplicationDate'].dt.month
loan_app_df['ApplicationWeekday'] = loan_app_df['LoanApplicationDate'].dt.weekday

loan_app_df['YearsBetweenRegAndLoan'] = (loan_app_df['LoanApplicationDate'].dt.year - loan_app_df['DateRegistered'].dt.year)

loan_app_df['AgeGroup'] = pd.cut(loan_app_df['CustomerAgeAtApplication'],
                        bins=[0, 25, 35, 50, 65, 100],
                        labels=['<25', '25–35', '35–50', '50–65', '65+'])

loan_app_df = pd.get_dummies(loan_app_df, columns=['AgeGroup'], prefix='AgeGroup')

loan_app_df.drop(['FirstName', 'MiddleName', 'LastName', 'Address'], axis=1, inplace=True)

customer_profile_features = loan_app_df.copy()

/var/folders/ft/6h9sv_z91731fk7nszwjtbnc0000gp/T/ipykernel_71532/4028608579.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loan_app_df['LoanApplicationDate'] = pd.to_datetime(loan_app_df['LoanApplicationDate'])


In [24]:
del cust_profile_df, loan_app_df

### remove unnecessary info 

In [25]:
customer_profile_features.drop(columns=['CustomerID','LoanApplicationDate','DateOfBirth','NumberOfDependents','LifecycleStatus','DateRegistered','Country','Defaulted','AvgRiskScore'],inplace=True)

## Transaction features

In [26]:
loan_app_df = df[['LoanId','CustomerID','LoanApplicationDate']]

In [27]:
transaction_features = []

# Iterate through each loan
for _, row in loan_app_df.iterrows():
    cust_id = row['CustomerID']
    app_date = row['LoanApplicationDate']
    
    # Filter transactions before the application date for the same customer
    txns = transaction_history[
        (transaction_history['CustomerID'] == cust_id) &
        (transaction_history['TransactionDate'] < app_date)
    ]

    if txns.empty:
        features = {
            'LoanId': row['LoanId'],
            'Txn_Count': 0,
            'Txn_Sum': 0,
            'Txn_Avg': 0,
            'Txn_Std': 0,
            'Txn_Max': 0,
            'Txn_Min': 0,
            'Txn_In_Count': 0,
            'Txn_Out_Count': 0,
            'Txn_Net': 0,
            'Txn_LastDaysAgo': None,
            'Txn_Sum_1M': 0,
            'Txn_Sum_3M': 0,
            'Txn_Sum_6M': 0
        }
    else:
        recent_1m = app_date - pd.Timedelta(days=30)
        recent_3m = app_date - pd.Timedelta(days=90)
        recent_6m = app_date - pd.Timedelta(days=180)

        incoming = txns[txns['Type'] == 'Incoming']
        outgoing = txns[txns['Type'] == 'Outgoing']

        features = {
            'LoanId': row['LoanId'],
            'Txn_Count': len(txns),
            'Txn_Sum': txns['Amount'].sum(),
            'Txn_Avg': txns['Amount'].mean(),
            'Txn_Std': txns['Amount'].std(),
            'Txn_Max': txns['Amount'].max(),
            'Txn_Min': txns['Amount'].min(),
            'Txn_In_Count': len(incoming),
            'Txn_Out_Count': len(outgoing),
            'Txn_Net': incoming['Amount'].sum() - outgoing['Amount'].sum(),
            'Txn_LastDaysAgo': (app_date - txns['TransactionDate'].max()).days,
            'Txn_Sum_1M': txns[txns['TransactionDate'] >= recent_1m]['Amount'].sum(),
            'Txn_Sum_3M': txns[txns['TransactionDate'] >= recent_3m]['Amount'].sum(),
            'Txn_Sum_6M': txns[txns['TransactionDate'] >= recent_6m]['Amount'].sum()
        }

    transaction_features.append(features)

transaction_features_df = pd.DataFrame(transaction_features)

## Add more features based on all data sources toghether 

In [28]:
merged_df = df.merge(customer_profile_features, on='LoanId').merge(transaction_features_df, on='LoanId')

In [29]:
merged_df

,LoanId,CustomerID,Amount,Target,LoanApplicationDate,Hist_CountLoans,Hist_TotalAmount,Hist_AvgAmount,Hist_StdAmount,Hist_MinAmount,...,Txn_Std,Txn_Max,Txn_Min,Txn_In_Count,Txn_Out_Count,Txn_Net,Txn_LastDaysAgo,Txn_Sum_1M,Txn_Sum_3M,Txn_Sum_6M
0,deacac44-4ed0-47d0-a7d9-b2ee5faf3204,CUST000000,9741.01,0,2021-08-14,0,0.00,0.000,0.000000,0.00,...,0.000000,0.00,0.00,0,0,0.00,NaN,0.00,0.00,0.00
1,27ffacb8-89b3-4868-a8fb-c2aec933992f,CUST000001,4526.59,0,2021-12-28,0,0.00,0.000,0.000000,0.00,...,0.000000,0.00,0.00,0,0,0.00,NaN,0.00,0.00,0.00
2,9f611adf-a663-4862-ba45-70624c50b357,CUST000001,3089.56,1,2023-04-05,1,4526.59,4526.590,0.000000,4526.59,...,0.000000,0.00,0.00,0,0,0.00,NaN,0.00,0.00,0.00
3,4b259bb7-4842-4364-a370-5fa7cbb68064,CUST000002,8068.27,0,2024-11-07,0,0.00,0.000,0.000000,0.00,...,69.594540,199.98,19.19,6,6,23.69,83.0,0.00,151.86,268.66
4,1ad36599-8e2e-483d-b0f2-9686bdd56d84,CUST000002,2109.99,1,2025-07-30,1,8068.27,8068.270,0.000000,8068.27,...,65.927696,199.98,19.19,8,8,-182.79,111.0,0.00,0.00,493.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6959,eeef2b40-15e7-4a09-b4ca-f4c12d88cecd,CUST004998,6802.84,0,2022-06-04,1,5082.12,5082.120,0.000000,5082.12,...,0.000000,0.00,0.00,0,0,0.00,NaN,0.00,0.00,0.00
6960,c6e4ac42-4ced-402b-b4dd-5f858a7fb9ff,CUST004998,3366.58,0,2023-04-05,2,11884.96,5942.480,1216.732781,5082.12,...,0.000000,0.00,0.00,0,0,0.00,NaN,0.00,0.00,0.00
6961,53741d01-06fb-4f33-8e36-214f7f9e17f9,CUST004999,272.55,0,2024-04-15,0,0.00,0.000,0.000000,0.00,...,49.655115,156.62,13.65,7,6,193.60,28.0,59.17,72.82,475.93
6962,2f519008-9544-4dd3-b529-0282f6f0cc05,CUST004999,7736.92,1,2024-05-17,1,272.55,272.550,0.000000,272.55,...,53.973386,173.06,13.65,8,7,86.50,15.0,239.02,298.19,714.95


In [30]:
##The term 1e-5 (which is 0.00001) is added to denominators in ratio calculations to avoid division by zero.
merged_df['IncomePerLoan'] = merged_df['Txn_Sum'] / (merged_df['Hist_CountLoans'] + 1e-5)
merged_df['TxnToLoanAmountRatio'] = merged_df['Txn_Sum'] / (merged_df['Hist_TotalAmount'] + 1e-5)
merged_df['NetToEMI'] = merged_df['Txn_Net'] / (merged_df['Hist_MeanEMIs'] + 1e-5)
merged_df['InflowToDependentsRatio'] = merged_df['Txn_Sum'] / (merged_df['CustomerAgeAtApplication'] + merged_df['PreviouslyDefaulted'] + 1)
merged_df['TxnToRegDaysRatio'] = merged_df['Txn_Count'] / (merged_df['DaysSinceRegistration'] + 1e-5)
merged_df['AvgTxnToAvgLoanRatio'] = merged_df['Txn_Avg'] / (merged_df['Hist_AvgAmount'] + 1e-5)

merged_df['IsTxnDeclining'] = ((merged_df['Txn_Sum_1M'] < merged_df['Txn_Sum_3M']) &
                               (merged_df['Txn_Sum_3M'] < merged_df['Txn_Sum_6M'])).astype(int)
merged_df['IsRecentLoanSpike'] = ((merged_df['Hist_DaysSinceLastLoan'] < 60) &
                                  (merged_df['Hist_CountLoans'] > 1)).astype(int)

merged_df['LowTxnHighLoans'] = ((merged_df['Txn_Sum'] < 100) &
                                (merged_df['Hist_TotalAmount'] > 10000)).astype(int)

merged_df['InactivitySpike'] = (merged_df['Txn_LastDaysAgo'].fillna(999) > 180).astype(int)

merged_df['TxnPerMonth'] = merged_df['Txn_Count'] / ((merged_df['DaysSinceRegistration'] / 30) + 1e-5)
merged_df['NetFlowPerTxn'] = merged_df['Txn_Net'] / (merged_df['Txn_Count'] + 1e-5)
merged_df['RecentActivityBoost'] = (merged_df['Txn_Sum_1M'] + 1) / (merged_df['Txn_Sum_6M'] + 1)
merged_df['RecentOutRatio'] = merged_df['Txn_Sum_1M'] / (merged_df['Txn_Sum'] + 1e-5)

merged_df['OutgoingToIncomingRatio'] = (merged_df['Txn_Out_Count'] + 1e-5) / (merged_df['Txn_In_Count'] + 1e-5)
merged_df['TxnStdToAvgRatio'] = merged_df['Txn_Std'] / (merged_df['Txn_Avg'] + 1e-5)
merged_df['EMIToTxnAvgRatio'] = merged_df['Hist_MeanEMIs'] / (merged_df['Txn_Avg'] + 1e-5)
merged_df['LoanAmountToTxnNetRatio'] = merged_df['Amount'] / (merged_df['Txn_Net'] + 1e-5)
merged_df['AgeToLoanRatio'] = merged_df['CustomerAgeAtApplication'] / (merged_df['Amount'] + 1e-5)
merged_df['TxnGrowth6to1M'] = (merged_df['Txn_Sum_1M'] + 1) / (merged_df['Txn_Sum_6M'] + 1)


In [32]:
merged_df.to_pickle('../data/final_features.pkl')